# Проект машинного обучения по датасету Amazon BestSelling Books 500

Задача: построить модели классификации для предсказания категории книги (`Fiction` / `Non-Fiction`) на основе признаков датасета `Amazon_BestSelling_Books_500.csv`.

Файл в задании имеет расширение `.xls`, но фактически читается как CSV-файл, поэтому используется `pd.read_csv`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
DATA_PATH = "Amazon_BestSelling_Books_500.xls"


## 1. Загрузка и первичный анализ данных

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates()

print("Размер датасета:", df.shape)
display(df.head())
display(df.info())


In [ ]:
display(df.describe(include="all").T)

missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame("missing_count"))


**Промежуточный вывод.**  
Датасет содержит сведения о книгах: ранг, название, автор, категория, поджанр, формат, цена, рейтинг, число отзывов, число недель в списке, издательство, год публикации, ISBN и Amazon BSR. Целевой признак `Category` содержит два класса, поэтому решается задача бинарной классификации.

## 2. Разведочный анализ данных и графики

In [ ]:
target_col = "Category"

plt.figure(figsize=(6, 4))
df[target_col].value_counts().plot(kind="bar")
plt.title("Распределение классов Category")
plt.xlabel("Категория")
plt.ylabel("Количество книг")
plt.xticks(rotation=0)
plt.show()

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
display(df[numeric_cols].head())


In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    df[col].hist(bins=25)
    plt.title(f"Распределение признака {col}")
    plt.xlabel(col)
    plt.ylabel("Количество")
    plt.show()


In [ ]:
cat_cols_for_plots = ["Sub-Genre", "Format", "Publisher"]

for col in cat_cols_for_plots:
    plt.figure(figsize=(9, 4))
    df[col].value_counts().head(12).plot(kind="bar")
    plt.title(f"Топ значений признака {col}")
    plt.xlabel(col)
    plt.ylabel("Количество")
    plt.xticks(rotation=45, ha="right")
    plt.show()


In [ ]:
for col in ["Price (USD)", "Rating", "Reviews", "Weeks on List", "Amazon BSR"]:
    plt.figure(figsize=(7, 4))
    df.boxplot(column=col, by=target_col)
    plt.title(f"{col} по категориям книг")
    plt.suptitle("")
    plt.xlabel("Категория")
    plt.ylabel(col)
    plt.show()


## 3. Анализ и заполнение пропусков

В исходном файле пропуски могут отсутствовать, но в проектном коде обработка пропусков всё равно добавлена:
- для числовых признаков используется медиана;
- для категориальных признаков используется наиболее частое значение.

Такой подход делает пайплайн устойчивым к появлению неполных данных.


In [ ]:
missing_report = df.isna().sum().to_frame("missing_count")
missing_report["missing_percent"] = (missing_report["missing_count"] / len(df) * 100).round(2)
display(missing_report)


## 4. Формирование вспомогательных признаков

In [ ]:
data = df.copy()

current_year = 2026
data["Book Age"] = current_year - data["Year Published"]
data["Reviews Log"] = np.log1p(data["Reviews"])
data["Price Per Rating"] = data["Price (USD)"] / data["Rating"].replace(0, np.nan)
data["BSR Log"] = np.log1p(data["Amazon BSR"])
data["Is Paperback"] = (data["Format"] == "Paperback").astype(int)

display(data.head())


## 5. Корреляционный анализ

In [ ]:
corr_cols = [
    "Rank", "Price (USD)", "Rating", "Reviews", "Weeks on List",
    "Year Published", "Amazon BSR", "Book Age", "Reviews Log",
    "Price Per Rating", "BSR Log", "Is Paperback"
]

corr = data[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Корреляционная матрица числовых признаков")
plt.show()

display(corr.round(3))


**Промежуточный вывод.**  
Для классификации можно использовать числовые признаки (`Rank`, `Price`, `Rating`, `Reviews`, `Weeks on List`, `Year Published`, `Amazon BSR`) и категориальные признаки (`Author`, `Sub-Genre`, `Format`, `Publisher`). Признаки `Title`, `ISBN`, `Amazon URL` исключаются, потому что они являются идентификаторами или текстовыми полями с высокой уникальностью и без простой прямой пользы для базовых моделей.

## 6. Выбор признаков, кодирование и масштабирование

In [ ]:
drop_cols = ["Category", "Title", "ISBN", "Amazon URL"]
X = data.drop(columns=drop_cols)
y = data[target_col]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)
print("Классы:", list(label_encoder.classes_))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 7. Выбор метрик качества

Используются четыре метрики:
1. `Accuracy` — общая доля правильных ответов.
2. `Balanced Accuracy` — учитывает качество по каждому классу и полезна при дисбалансе классов.
3. `F1-macro` — усредняет F1 по классам и показывает баланс между precision и recall.
4. `ROC AUC` — оценивает способность модели разделять два класса по вероятностной оценке.


## 8. Формирование обучающей и тестовой выборок

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## 9. Выбор моделей

Используются не менее пяти моделей, включая две ансамблевые:
- DummyClassifier — простейшая модель для контроля baseline;
- LogisticRegression;
- KNeighborsClassifier;
- SVC;
- DecisionTreeClassifier;
- RandomForestClassifier — ансамблевая модель;
- GradientBoostingClassifier — ансамблевая модель.


In [ ]:
models = {
    "Dummy": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}


In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro")
    }

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        result["roc_auc"] = roc_auc_score(y_test, y_score)
    else:
        result["roc_auc"] = np.nan

    return result

baseline_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    metrics = evaluate_model(pipe, X_train, X_test, y_train, y_test)
    baseline_results.append({"model": name, **metrics})

baseline_df = pd.DataFrame(baseline_results).sort_values(by="f1_macro", ascending=False)
display(baseline_df)


In [ ]:
metric_cols = ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]

for metric in metric_cols:
    plt.figure(figsize=(8, 4))
    temp = baseline_df.sort_values(metric)
    plt.barh(temp["model"], temp[metric])
    plt.title(f"Baseline-модели: {metric}")
    plt.xlabel(metric)
    plt.show()


## 10. Подбор гиперпараметров с GridSearchCV

In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10],
        "model__solver": ["lbfgs"]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 11],
        "model__weights": ["uniform", "distance"]
    },
    "SVC": {
        "model__C": [0.5, 1, 5],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    },
    "Decision Tree": {
        "model__max_depth": [None, 3, 5, 8, 12],
        "model__min_samples_split": [2, 5, 10]
    },
    "Random Forest": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5]
    },
    "Gradient Boosting": {
        "model__n_estimators": [50, 100, 150],
        "model__learning_rate": [0.03, 0.1, 0.2],
        "model__max_depth": [2, 3]
    }
}

tuned_models = {}
tuned_results = []

for name, grid in param_grids.items():
    print(f"Подбор гиперпараметров: {name}")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", models[name])
    ])

    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="f1_macro",
        cv=5,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_

    metrics = evaluate_model(search.best_estimator_, X_train, X_test, y_train, y_test)
    tuned_results.append({
        "model": name,
        "best_params": search.best_params_,
        **metrics
    })

tuned_df = pd.DataFrame(tuned_results).sort_values(by="f1_macro", ascending=False)
display(tuned_df)


## 11. Сравнение baseline и моделей после подбора гиперпараметров

In [ ]:
baseline_compare = baseline_df[baseline_df["model"] != "Dummy"].copy()
baseline_compare["stage"] = "baseline"

tuned_compare = tuned_df.drop(columns=["best_params"]).copy()
tuned_compare["stage"] = "tuned"

compare_df = pd.concat([baseline_compare, tuned_compare], ignore_index=True)
display(compare_df.sort_values(["model", "stage"]))


In [ ]:
for metric in metric_cols:
    pivot = compare_df.pivot(index="model", columns="stage", values=metric)

    plt.figure(figsize=(9, 4))
    x = np.arange(len(pivot.index))
    width = 0.35

    plt.bar(x - width / 2, pivot["baseline"], width, label="baseline")
    plt.bar(x + width / 2, pivot["tuned"], width, label="tuned")

    plt.xticks(x, pivot.index, rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(f"Сравнение baseline и tuned по метрике {metric}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 12. Матрица ошибок лучшей модели

In [ ]:
best_model_name = tuned_df.iloc[0]["model"]
best_model = tuned_models[best_model_name]

print("Лучшая модель:", best_model_name)
print("Параметры:", tuned_df.iloc[0]["best_params"])

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot()
plt.title(f"Матрица ошибок: {best_model_name}")
plt.show()


## 13. График влияния гиперпараметра на качество модели

In [ ]:
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])

param_range = [10, 25, 50, 100, 150, 200, 300]

train_scores, valid_scores = validation_curve(
    rf_pipe,
    X_train,
    y_train,
    param_name="model__n_estimators",
    param_range=param_range,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

plt.figure(figsize=(7, 4))
plt.plot(param_range, train_scores.mean(axis=1), marker="o", label="train")
plt.plot(param_range, valid_scores.mean(axis=1), marker="o", label="validation")
plt.title("Влияние n_estimators на F1-macro для Random Forest")
plt.xlabel("n_estimators")
plt.ylabel("F1-macro")
plt.legend()
plt.grid(True)
plt.show()


## 14. Кривые обучения

In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    best_model,
    X_train,
    y_train,
    train_sizes=np.linspace(0.2, 1.0, 5),
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="train")
plt.plot(train_sizes, valid_scores.mean(axis=1), marker="o", label="validation")
plt.title(f"Кривая обучения: {best_model_name}")
plt.xlabel("Размер обучающей выборки")
plt.ylabel("F1-macro")
plt.legend()
plt.grid(True)
plt.show()


## 15. Итоговые выводы

1. Для датасета Amazon BestSelling Books 500 решена задача бинарной классификации: предсказание категории книги `Fiction` или `Non-Fiction`.
2. Данные содержат как числовые, так и категориальные признаки, поэтому использован общий пайплайн с заполнением пропусков, масштабированием числовых признаков и One-Hot Encoding для категориальных признаков.
3. Для оценки качества выбраны `Accuracy`, `Balanced Accuracy`, `F1-macro` и `ROC AUC`.
4. Построено несколько моделей, включая ансамблевые `Random Forest` и `Gradient Boosting`.
5. Подбор гиперпараметров через `GridSearchCV` позволяет сравнить качество моделей до и после настройки.
6. Итоговая модель выбирается по `F1-macro`, так как эта метрика учитывает качество классификации по обоим классам.
